# 🇸🇳 AIMS GAAI Exam — Wolof Assistant Fine-Tuning
**Course:** Applied Generative and Agentic AI — Dr. Papa-Séga WADE  
**Model:** Qwen3-0.6B + LoRA (assistant-only -100 masking)  
**Data:** 3 sources — AYA, Soynade, Synthetic Wolof  

> ⚡ Make sure Runtime → Change runtime type → **T4 GPU** is selected before running!

## ✅ Step 0 — Check GPU

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only — change runtime!')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory/1e9, 1), 'GB' if torch.cuda.is_available() else '')

## 📦 Step 1 — Install dependencies

In [ ]:
%%capture
!pip install transformers>=4.45 accelerate>=0.33 datasets>=2.20 peft>=0.12 trl>=0.10 \
    pandas>=2.0 typer>=0.12 huggingface_hub>=0.24 tensorboard>=2.16 \
    matplotlib>=3.8 PyYAML>=6.0 evaluate sacrebleu rouge_score
print('✅ Installation complete')

## 🔐 Step 2 — Hugging Face Login

In [ ]:
# Paste your HF Write token here
HF_TOKEN = "hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX"  # <-- remplace par ton token
HF_USERNAME = "niangmariame513"
MODEL_NAME = "wolof-assistant-qwen3"
REPO_ID = f"{HF_USERNAME}/{MODEL_NAME}"

from huggingface_hub import login
login(token=HF_TOKEN)
print(f'✅ Logged in as: {HF_USERNAME}')
print(f'✅ Model will be pushed to: {REPO_ID}')

## 📁 Step 3 — Clone the exam repository

In [ ]:
import os

REPO_URL = "https://github.com/papasega/aims_cases_studies_finetuning_exam"
PROJECT_DIR = "/content/aims_exam"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
else:
    print('Repo already cloned')

os.chdir(PROJECT_DIR)
!ls

## 📊 Step 4 — Prepare data (3 sources)

In [ ]:
import subprocess, os
os.environ['HF_TOKEN'] = HF_TOKEN

!python src/download_datasets.py all \
    --max-aya-examples 500 \
    --max-soynade-examples 500 \
    --validation-ratio 0.05 \
    --eval-ratio 0.05

print('\n✅ Data sources prepared:')
!ls data/
print('\n✅ Splits:')
!ls data/splits/

In [ ]:
# Verify all 3 sources are present
import json

sources = {
    'AYA (general)': 'data/splits/chat_aya_train.json',
    'Soynade (domain)': 'data/splits/chat_soynade_train.json',
    'Synthetic (instructions)': 'data/splits/chat_synth_train.json'
}

for name, path in sources.items():
    with open(path) as f:
        data = json.load(f)
    print(f'✅ {name}: {len(data)} training examples')

with open('data/splits/eval_all.jsonl') as f:
    eval_count = sum(1 for _ in f)
print(f'\n✅ Eval set: {eval_count} examples')

## 🏋️ Step 5 — Train the LoRA adapter

> This uses assistant-only `-100` masking: system/user tokens are ignored, only assistant tokens are learned.

In [ ]:
# Full training on GPU — should take ~20-40 min on T4
!python train_lora_assistant_only.py \
    --config configs/wolof_training_config.yml \
    --model-choice qwen

In [ ]:
# Check that adapter was saved
!ls outputs/qwen3_0_6b_wolof_lora/
print('')
!ls outputs/qwen3_0_6b_wolof_lora/best_adapter/ 2>/dev/null || echo 'best_adapter not found, checking latest...'
!ls outputs/qwen3_0_6b_wolof_lora/latest_adapter/ 2>/dev/null

## 📈 Step 6 — Evaluate (BLEU, ROUGE, F1, Exact Match)

In [ ]:
!python evaluation.py \
    --data data/splits/eval_all.jsonl \
    --generate \
    --model-choice qwen \
    --adapter auto

## 💬 Step 7 — Test inference locally

In [ ]:
!python inference_lora_qwen3_wolof.py \
    --model-choice qwen \
    --adapter auto \
    --category education \
    --instruction "Ana nga?"

In [ ]:
!python inference_lora_qwen3_wolof.py \
    --model-choice qwen \
    --adapter auto \
    --category education \
    --instruction "Yal na Yàlla baal ma."

## 🚀 Step 8 — Push adapter to Hugging Face Hub

In [ ]:
!python push_to_hub.py \
    --repo-id {REPO_ID} \
    --model-choice qwen \
    --adapter-dir auto \
    --checkpoint best \
    --public

print(f'\n✅ Model pushed to: https://huggingface.co/{REPO_ID}')

## 📋 Step 9 — Save evaluation results for your report

In [ ]:
import os, glob

# Show all output files
print('=== Output files ===')
for f in glob.glob('outputs/**/*', recursive=True):
    if os.path.isfile(f) and not f.endswith(('.bin', '.safetensors', '.pt')):
        print(f)

print('\n=== Training curves ===')
for f in glob.glob('outputs/**/*.json', recursive=True):
    print(f)

In [ ]:
# Plot training loss curve
import json, matplotlib.pyplot as plt, glob

history_files = glob.glob('outputs/**/training_history.json', recursive=True)
if history_files:
    with open(history_files[0]) as f:
        history = json.load(f)
    
    if 'train_loss' in history:
        plt.figure(figsize=(10,4))
        plt.subplot(1,2,1)
        plt.plot(history['train_loss'], label='Train Loss')
        plt.title('Training Loss')
        plt.xlabel('Step')
        plt.ylabel('Loss')
        plt.legend()
        
        if 'eval_loss' in history:
            plt.subplot(1,2,2)
            plt.plot(history['eval_loss'], label='Eval Loss', color='orange')
            plt.title('Validation Loss')
            plt.xlabel('Epoch')
            plt.legend()
        
        plt.tight_layout()
        plt.savefig('outputs/training_curves.png', dpi=150)
        plt.show()
        print('✅ Curves saved to outputs/training_curves.png')
else:
    print('No training history file found yet')

## 📥 Step 10 — Download adapter for local use (optional)

In [ ]:
# Zip the best adapter to download
import shutil

adapter_dir = 'outputs/qwen3_0_6b_wolof_lora/best_adapter'
if not os.path.exists(adapter_dir):
    adapter_dir = 'outputs/qwen3_0_6b_wolof_lora/latest_adapter'

shutil.make_archive('/content/wolof_lora_adapter', 'zip', adapter_dir)
print('✅ Adapter zipped: /content/wolof_lora_adapter.zip')
print('Download it from the Files panel on the left (folder icon)')

---
## 🎉 Summary

After running all cells, you should have:

| Item | Status |
|------|--------|
| 3 data sources prepared | ✅ |
| LoRA adapter trained with -100 masking | ✅ |
| Evaluation metrics (BLEU, ROUGE, F1) | ✅ |
| Model pushed to HF Hub | ✅ |

**Next step:** Deploy the Hugging Face Space using `hf_space/app.py`

**Your model:** https://huggingface.co/niangmariame513/wolof-assistant-qwen3